# Connecting the Dots — Phase 3 Training
## Run on Google Colab (Free T4 GPU)
**Steps:** Setup → Upload dataset → Train ArcFace → Train Age GAN → Download weights

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────
!pip install -q mtcnn tqdm pyyaml opencv-python-headless scikit-learn matplotlib

In [ ]:
# ── 3. Mount Google Drive (to save checkpoints) ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/connecting_the_dots'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

In [ ]:
# ── 4. Upload your code ───────────────────────────────────────────────────
# Option A: Upload zip from your computer
from google.colab import files
print('Upload connecting-the-dots.zip from your computer:')
uploaded = files.upload()

In [ ]:
# Option B: Clone from GitHub (if you pushed to a repo)
# !git clone https://github.com/YOUR_USERNAME/connecting-the-dots.git

# Extract zip
!unzip -q connecting-the-dots.zip
%cd connecting-the-dots/ml/training
!ls

In [ ]:
# ── 5. Download LFW dataset (for quick test) ──────────────────────────────
!python dataset_downloader.py --dataset lfw --output /content/data
# Then align:
!python dataset_downloader.py --align \
    --input  /content/data/lfw/lfw-funneled \
    --output /content/data/aligned \
    --img-size 112
# Split train/val:
!python dataset_downloader.py --split \
    --input  /content/data/aligned \
    --output /content/data/split \
    --ratio  0.9
# Generate verification pairs:
!python dataset_downloader.py --pairs \
    --input      /content/data/split/val \
    --output     /content/data \
    --num-pairs  3000

In [ ]:
# ── 6. Train ArcFace ─────────────────────────────────────────────────────
!python train_arcface.py \
    --data_dir   /content/data/split \
    --output_dir /content/drive/MyDrive/connecting_the_dots/arcface \
    --backbone   resnet50 \
    --epochs     50 \
    --batch_size 64

In [ ]:
# ── 7. Evaluate the trained model ────────────────────────────────────────
!python evaluate_model.py \
    --checkpoint /content/drive/MyDrive/connecting_the_dots/arcface/best.pth \
    --pairs      /content/data/pairs.json \
    --output_dir /content/eval_results

# Show the ROC curve
from IPython.display import Image
Image('/content/eval_results/roc_curve.png')

In [ ]:
# ── 8. Download UTKFace and train Age GAN ────────────────────────────────
# Upload UTKFace zip (downloaded manually from susanqq.github.io/UTKFace/)
uploaded = files.upload()   # upload UTKFace_part1.tar.gz etc.
!tar xzf UTKFace_part1.tar.gz -C /content/data/utk/ 2>/dev/null || true

# Prepare for GAN
!python dataset_downloader.py --utk-prep \
    --input  /content/data/utk \
    --output /content/data/utk_grouped \
    --img-size 256

In [ ]:
# Train Age GAN
!python train_age_gan.py \
    --data_dir   /content/data/utk_grouped \
    --output_dir /content/drive/MyDrive/connecting_the_dots/age_gan \
    --epochs     100 \
    --batch_size 16

In [ ]:
# ── 9. View sample outputs from Age GAN ─────────────────────────────────
import glob
from IPython.display import Image
samples = sorted(glob.glob('/content/drive/MyDrive/connecting_the_dots/age_gan/samples/*.jpg'))
if samples:
    print(f'Latest sample: {samples[-1]}')
    display(Image(samples[-1]))

In [ ]:
# ── 10. Download trained weights ─────────────────────────────────────────
# They're already in Google Drive, but you can also download directly:
files.download('/content/drive/MyDrive/connecting_the_dots/arcface/best.pth')
files.download('/content/drive/MyDrive/connecting_the_dots/age_gan/generator_final.pth')